# Analisis de sentimientos
Es el proceso automatizado de etiquetar datos de acuerdo al sentimiento del texto como: positivo, negativo, neutro. Es usado en compañías para detectar nuevas percepciones y conocimientos de datos en escala. En su esencia es lenguaje natural de procesamiento, una técnica para clasificar la polaridad de texto segun su sentimiento.

Permite el procesamiento de datos en escala y tiempo real, se puede automatizar esta clasificación para entender directamente como las personas hablan y opinan acerca de un tema en específico, lo que permite a organizaciones:
- Tomar acciones basada en el conocimiento adquerido a partir de los datos.
- Entender como las personas estan hablando de una marca vs competidores 
- Analizar comentarios de encuestas y reseñas de productos 
- Analizar solicitudes o reportes de errores en servicios (tickets) para evitar tasa de cancelación (churn)

# Comandos relevantes para el proyecto en la terminal:
### Instalación y configuración  



In [ ]:
# En la terminal con el env. ya creado, versión usada para este proyecto python=3.11
#(sentiment_analysis)  > conda env export --no-builds > environment.yml 
#(sentiment_analysis)  > 
#(sentiment_analysis)  > conda install -c conda-forge transformers
# Instalar
#(sentiment_analysis)  > conda install -c conda-forge ipykernel
# Registrar
#(sentiment_analysis)  > python -m ipykernel install --user --name mi_entorno --display-name "Python (mi_entorno)" 
#CPU
#(sentiment_analysis)  > conda install pytorch torchvision torchaudio cpuonly -c pytorch
#GPU
#(sentiment_analysis)  > conda install pytorch torchvision torchaudio pytorch-cuda=12.1 -c pytorch -c nvidia
# Forge
#(sentiment_analysis)  > conda install -c conda-forge ipywidgets pandas

In [1]:
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer as sia
import matplotlib.pyplot as plt
from deep_translator import GoogleTranslator as ggt
import seaborn as sns
from wordcloud import WordCloud as wc
from textblob import TextBlob  


In [2]:
#datos = pd.read_csv('comentarios.csv', sep=';', encoding='latin-1')

# Se utiliza "latin-1" debido a que el archivo posee caracteres como tildes (á, é etc) o la letra eñe (ñ) que el tradicional utf-8 no soporta 
#Limpieza
filas = []
with open('comentarios.csv',encoding='latin1' ) as f:   # with: Context manager, cerrar el archivo 
    header = next(f)         # Salta header (primera columna de f)
    
    for linea in f:
        # .strip -> elimina espacios blanco y \n  
        linea = linea.strip()

        if linea.startswith('"'):
            linea = linea.strip(',')
            linea = linea.strip('"')
        else:
            linea = linea.strip(',')

        parte_linea = linea.split(';')      #.split(';') separa en pedazos
        texto = ';'.join(parte_linea[:-2])          # parte_linea[:-2] desde el inicio hasta el penultimo elemento, todo excepto los ultimos 2 elementos
        #texto = ';'
        sentimiento = parte_linea[-2].strip()
        categoria = parte_linea[-1].strip()
        filas.append([texto,sentimiento,categoria])

comentarios = pd.DataFrame(filas, columns=['texto', 'sentimiento', 'categoria'])
comentarios['sentimiento'] = pd.to_numeric(comentarios['sentimiento'], errors='coerce') # default error raise -> Exception 
comentarios['categoria']   = pd.to_numeric(comentarios['categoria'],   errors='coerce') # Invalid converts into Nan

#comentarios


#with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', None): 
#    display(comentarios)
display(comentarios)


,texto,sentimiento,categoria
0,Excelente atención y muy buena disposición en ...,1,1
1,Los productos son de alta calidad y siempre ll...,1,1
2,Me encanta el diseño y la funcionalidad de la ...,1,1
3,"Estoy muy contento con el producto, superó mis...",1,1
4,"El servicio postventa fue excelente, resolvier...",1,1
...,...,...,...
198,El documental que vi sobre la historia del art...,0,2
199,"La visita al museo fue frustrante, muchas exhi...",0,2
200,Los talleres extracurriculares han fomentado e...,1,3
201,La integración de tecnología en el aula ha fac...,1,3


In [ ]:
traductor = ggt(source='auto',target='en')

polaridad = []
subjetividad = []
sentimiento_tb = []

for texto in comentarios['texto']:
    traduccion = traductor.translate(texto)

    blob = TextBlob(traduccion)
    pol = blob.sentiment.polarity
    sub = blob.sentiment.subjectivity

    if pol > 0:
        etiqueta = "Positivo"
    elif pol < 0:
        etiqueta = "Negativo"
    else:
        etiqueta = "Neutro"
    
    polaridad.append(pol)
    subjetividad.append(sub)
    sentimiento_tb.append(etiqueta)

comentarios['tb_polaridad'] = polaridad
comentarios['tb_subjetividad'] = subjetividad
comentarios['tb_sentimiento'] = sentimiento_tb

comentarios.head(10)

,texto,sentimiento,categoria,tb_polaridad,tb_subjetividad,tb_sentimiento
0,Excelente atención y muy buena disposición en ...,1,1,0.955000,0.890000,Neutro
1,Los productos son de alta calidad y siempre ll...,1,1,0.405000,0.770000,Neutro
2,Me encanta el diseño y la funcionalidad de la ...,1,1,0.466667,0.716667,Neutro
3,"Estoy muy contento con el producto, superó mis...",1,1,1.000000,1.000000,Neutro
4,"El servicio postventa fue excelente, resolvier...",1,1,1.000000,1.000000,Neutro
5,"Muy mala experiencia con la compra, el product...",0,1,-0.655000,0.733333,Negativo
6,El servicio al cliente fue poco profesional y ...,0,1,0.000000,0.000000,Neutro
7,"Me siento decepcionado, la calidad no es como ...",0,1,-0.750000,0.750000,Negativo
8,Tardaron demasiado en responder y no soluciona...,0,1,-0.050000,0.400000,Negativo
9,"No recomendaría este servicio, tuve problemas ...",0,1,0.000000,0.000000,Neutro


In [8]:
print(type(comentarios['tb_polaridad'][0]))
print(comentarios['tb_polaridad'][0] > 0)
print(comentarios['tb_sentimiento'].value_counts())



<class 'numpy.float64'>
True
tb_sentimiento
Neutro      136
Negativo     67
Name: count, dtype: int64


## Bibliografía:
1. https://huggingface.co/blog/sentiment-analysis-python
2.  
3. 